# Data Preparation

This notebook prepares the classification dataset and saves everything needed for PyTorch training:

- Cleaned train / validation / test CSV files
- BCU CSV file
- Scaled NumPy arrays for train / validation / test / BCU
- Label arrays for train / validation / test


In [1]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from astropy.table import Table
from sklearn.compose import make_column_transformer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

sns.set_theme(style="darkgrid")


In [2]:
np.random.seed(0)


In [3]:
DATA_DIR = Path("./classification_data")
DATA_DIR.mkdir(exist_ok=True)

SELECTED_COLUMNS = [
    "PL_Index",
    "nu_syn",
    "LP_Index",
    "Pivot_Energy",
    "Frac_Variability",
    "Variability_Index",
    "nuFnu_syn",
    "CLASS",
]

TARGET = "CLASS"

FEATURE_COLUMNS = [column for column in SELECTED_COLUMNS if column != TARGET]
LOG_SKIP_COLUMNS = {
    "Frac_Variability",
    "GLAT",
    "GLON",
    "LP_Index",
    "LP_beta",
    "PL_Index",
    "Unc_Flux1000",
    "Unc_PL_Index",
}

def normalize_class_value(value):
    if isinstance(value, (bytes, bytearray)):
        value = value.decode("utf-8")
    return str(value).upper()

def load_classification_dataframe():
    df_h = Table.read("./table-4LAC-DR3-h.fits").to_pandas()
    df_l = Table.read("./table-4LAC-DR3-l.fits").to_pandas()

    df_h = df_h[df_h["Flags"] == 0]
    df_l = df_l[df_l["Flags"] == 0]

    classification_df = pd.concat([df_h, df_l], ignore_index=True)
    classification_df["CLASS"] = classification_df["CLASS"].apply(normalize_class_value)
    classification_df = classification_df[classification_df["CLASS"].isin(["BLL", "FSRQ", "BCU"])].copy()
    classification_df = classification_df[SELECTED_COLUMNS].copy()
    return classification_df

classification_df = load_classification_dataframe()
classification_df.head()


,PL_Index,nu_syn,LP_Index,Pivot_Energy,Frac_Variability,Variability_Index,nuFnu_syn,CLASS
0,2.271696,1.000000e+14,2.254081,2420.575195,0.675882,25.313953,3.716456e-13,BCU
1,2.116692,9.120108e+13,2.078927,1721.603882,0.406565,46.780693,1.936770e-12,BLL
2,1.939160,3.609939e+12,1.661223,4307.082520,0.000000,9.272764,2.473256e-13,BLL
3,2.654060,1.584893e+14,2.514159,329.773743,0.996138,1910.935791,1.152471e-12,FSRQ
4,1.876663,1.659588e+13,1.716620,4429.934082,0.902851,24.557972,9.716011e-13,BCU


In [4]:
def plot_column_distribution(column=None, df=None, transformed=False):
    plt.figure(figsize=(6, 4))
    plt.hist(df[column], bins=100)
    plt.ylabel("Frequency")
    plt.xlabel("Values")
    if transformed:
        plt.title(f"Distribution of {column} after transformation")
    else:
        plt.title(f"Distribution of {column}")
    plt.show()


for column in FEATURE_COLUMNS:
    if column in LOG_SKIP_COLUMNS:
        continue

    if column == "nuFnu_syn":
        classification_df[column] = np.log(classification_df[column] + 1e-12)
    elif column == "nu_syn":
        classification_df[column] = np.log(classification_df[column] + 1e12)
    else:
        classification_df[column] = np.log(classification_df[column])

classification_df.describe(include="all")


,PL_Index,nu_syn,LP_Index,Pivot_Energy,Frac_Variability,Variability_Index,nuFnu_syn,CLASS
count,3120.000000,3120.000000,3120.000000,3120.000000,3120.000000,3120.000000,3120.000000,3120
unique,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3
top,NaN,NaN,NaN,NaN,NaN,NaN,NaN,BLL
freq,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1335
mean,2.189187,31.148043,2.098539,7.534712,0.413246,3.481265,-26.899857,NaN
std,0.287124,3.268656,0.344169,0.800561,0.383216,1.431227,0.738055,NaN
min,1.418864,27.631021,-0.048980,5.688085,0.000000,0.470044,-27.631021,NaN
25%,1.971043,28.580706,1.868027,6.915492,0.053641,2.546236,-27.457013,NaN
50%,2.181250,30.261650,2.097951,7.491702,0.357368,3.044733,-27.079242,NaN
75%,2.400465,33.152597,2.333431,8.094847,0.613080,4.029697,-26.561881,NaN


In [5]:
bcu_samples_df = classification_df[classification_df[TARGET] == "BCU"].copy()
labeled_df = classification_df[classification_df[TARGET] != "BCU"].copy()
labeled_df[TARGET] = labeled_df[TARGET].map({"BLL": 0, "FSRQ": 1}).astype(np.int64)

features = labeled_df.drop(columns=[TARGET])
labels = labeled_df[TARGET]

X_train_df, X_test_df, y_train_series, y_test_series = train_test_split(
    features,
    labels,
    test_size=0.20,
    random_state=42    
)

X_val_df, X_test_df, y_val_series, y_test_series = train_test_split(
    X_test_df,
    y_test_series,
    test_size=0.50,
    random_state=42
)

print("train:", X_train_df.shape, y_train_series.shape)
print("val:", X_val_df.shape, y_val_series.shape)
print("test:", X_test_df.shape, y_test_series.shape)
print("bcu:", bcu_samples_df.shape)


train: (1604, 7) (1604,)
val: (200, 7) (200,)
test: (201, 7) (201,)
bcu: (1115, 8)


In [6]:
transformer = make_column_transformer((StandardScaler(), FEATURE_COLUMNS))
transformer.fit(X_train_df)

X_train = transformer.transform(X_train_df).astype(np.float32)
X_val = transformer.transform(X_val_df).astype(np.float32)
X_test = transformer.transform(X_test_df).astype(np.float32)
bcu_samples = transformer.transform(bcu_samples_df[FEATURE_COLUMNS]).astype(np.float32)

y_train = y_train_series.to_numpy(dtype=np.int64)
y_val = y_val_series.to_numpy(dtype=np.int64)
y_test = y_test_series.to_numpy(dtype=np.int64)

X_train_df.to_csv(DATA_DIR / "train_samples.csv", index=False)
X_val_df.to_csv(DATA_DIR / "val_samples.csv", index=False)
X_test_df.to_csv(DATA_DIR / "test_samples.csv", index=False)
y_train_series.to_frame(name=TARGET).to_csv(DATA_DIR / "train_labels.csv", index=False)
y_val_series.to_frame(name=TARGET).to_csv(DATA_DIR / "val_labels.csv", index=False)
y_test_series.to_frame(name=TARGET).to_csv(DATA_DIR / "test_labels.csv", index=False)
bcu_samples_df.to_csv(DATA_DIR / "bcu_samples.csv", index=False)

np.save(DATA_DIR / "X_train.npy", X_train)
np.save(DATA_DIR / "X_val.npy", X_val)
np.save(DATA_DIR / "X_test.npy", X_test)
np.save(DATA_DIR / "y_train.npy", y_train)
np.save(DATA_DIR / "y_val.npy", y_val)
np.save(DATA_DIR / "y_test.npy", y_test)
np.save(DATA_DIR / "bcu_samples.npy", bcu_samples)

metadata = {
    "selected_columns": SELECTED_COLUMNS,
    "feature_columns": FEATURE_COLUMNS,
    "target": TARGET,
    "train_shape": list(X_train.shape),
    "val_shape": list(X_val.shape),
    "test_shape": list(X_test.shape),
    "bcu_shape": list(bcu_samples.shape),
}
(DATA_DIR / "metadata.json").write_text(json.dumps(metadata, indent=2))

metadata


{'selected_columns': ['PL_Index',
  'nu_syn',
  'LP_Index',
  'Pivot_Energy',
  'Frac_Variability',
  'Variability_Index',
  'nuFnu_syn',
  'CLASS'],
 'feature_columns': ['PL_Index',
  'nu_syn',
  'LP_Index',
  'Pivot_Energy',
  'Frac_Variability',
  'Variability_Index',
  'nuFnu_syn'],
 'target': 'CLASS',
 'train_shape': [1604, 7],
 'val_shape': [200, 7],
 'test_shape': [201, 7],
 'bcu_shape': [1115, 7]}

In [7]:
X_train = np.load(DATA_DIR / "X_train.npy")
X_val = np.load(DATA_DIR / "X_val.npy")
X_test = np.load(DATA_DIR / "X_test.npy")
y_train = np.load(DATA_DIR / "y_train.npy")
y_val = np.load(DATA_DIR / "y_val.npy")
y_test = np.load(DATA_DIR / "y_test.npy")
bcu_samples = np.load(DATA_DIR / "bcu_samples.npy")

print("train:", X_train.shape, y_train.shape)
print("val:", X_val.shape, y_val.shape)
print("test:", X_test.shape, y_test.shape)
print("bcu:", bcu_samples.shape)


train: (1604, 7) (1604,)
val: (200, 7) (200,)
test: (201, 7) (201,)
bcu: (1115, 7)
